# 04 — Market Basket Analysis

Invoice-product preparation, frequent itemsets, and association rules.

## 1. Objective

Identify products that frequently occur together in the same invoice.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = Path.cwd().parent

REPORTS = ROOT / "reports"
PROCESSED = ROOT / "data" / "processed"
FIGURES = REPORTS / "figures"

print("Project root:", ROOT)
print("Reports directory exists:", REPORTS.exists())
print("Processed data directory exists:", PROCESSED.exists())

Project root: C:\Users\tadij\Documents\retail-customer-intelligence-python
Reports directory exists: True
Processed data directory exists: True


## 2. Transaction preparation

In [2]:
basket_path = PROCESSED / "market_basket_transactions.csv"
basket = pd.read_csv(basket_path, dtype={"invoiceno": "string", "stockcode": "string"})

print("Prepared rows:", len(basket))
print("Unique invoices:", basket["invoiceno"].nunique())
print("Unique products:", basket["stockcode"].nunique())
display(basket.head())

Prepared rows: 517478
Unique invoices: 19792
Unique products: 3809


## 3. Frequent itemsets

In [3]:
itemsets = pd.read_csv(REPORTS / "frequent_itemsets.csv")
display(itemsets.head(10))

,items,itemset_size,support
0,85123A,1,0.114440
1,85099B,1,0.105548
2,22423,1,0.100445
3,47566,1,0.085135
4,20725,1,0.079072
5,84879,1,0.073515
6,22197,1,0.070331
7,22720,1,0.069978
8,21212,1,0.066694
9,22383,1,0.064875


## 4. Association rules

In [4]:
rules_path = REPORTS / "market_basket_rules_enhanced.csv"
rules = pd.read_csv(rules_path)
display(rules.head(10))

,antecedents,antecedent_descriptions,consequents,consequent_descriptions,support,supporting_invoice_count,confidence,lift,leverage,conviction
0,22916,22916 (HERB MARKER THYME),22917 | 22918,22917 (HERB MARKER ROSEMARY) + 22918 (HERB MAR...,0.010257,203,0.856540,78.849495,0.010127,6.894867
1,22917 | 22918,22917 (HERB MARKER ROSEMARY) + 22918 (HERB MAR...,22916,22916 (HERB MARKER THYME),0.010257,203,0.944186,78.849495,0.010127,17.702123
2,22917,22917 (HERB MARKER ROSEMARY),22916 | 22918,22916 (HERB MARKER THYME) + 22918 (HERB MARKER...,0.010257,203,0.845833,78.594992,0.010126,6.416679
3,22916 | 22918,22916 (HERB MARKER THYME) + 22918 (HERB MARKER...,22917,22917 (HERB MARKER ROSEMARY),0.010257,203,0.953052,78.594992,0.010126,21.041714
4,22916 | 22920,22916 (HERB MARKER THYME) + 22920 (HERB MARKER...,22917,22917 (HERB MARKER ROSEMARY),0.010105,200,0.952381,78.539683,0.009976,20.745352
5,22917,22917 (HERB MARKER ROSEMARY),22916 | 22920,22916 (HERB MARKER THYME) + 22920 (HERB MARKER...,0.010105,200,0.833333,78.539683,0.009976,5.936338
6,22917 | 22920,22917 (HERB MARKER ROSEMARY) + 22920 (HERB MAR...,22916,22916 (HERB MARKER THYME),0.010105,200,0.934579,78.047242,0.009976,15.102675
7,22916,22916 (HERB MARKER THYME),22917 | 22920,22917 (HERB MARKER ROSEMARY) + 22920 (HERB MAR...,0.010105,200,0.843882,78.047242,0.009976,6.336147
8,22916,22916 (HERB MARKER THYME),22917,22917 (HERB MARKER ROSEMARY),0.011166,221,0.932489,76.899297,0.011021,14.632882
9,22917,22917 (HERB MARKER ROSEMARY),22916,22916 (HERB MARKER THYME),0.011166,221,0.920833,76.899297,0.011021,12.480322


## 5. Interpreting rule metrics

- Support: fraction of invoices containing the combined itemset.
- Confidence: fraction of antecedent invoices that also contain the consequent.
- Lift: observed co-occurrence relative to the consequent's baseline frequency.

Association does not establish causation. Low-support rules should be interpreted cautiously.

In [5]:
metric_columns = [
    c for c in ["support", "confidence", "lift", "supporting_invoice_count"]
    if c in rules.columns
]
display(rules[metric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
support,1676.0,0.014107,0.004624,0.010004,0.010863,0.012581,0.015713,0.041684
confidence,1676.0,0.514447,0.152116,0.300000,0.388283,0.491586,0.617008,0.953052
lift,1676.0,13.275659,11.497965,2.653348,7.523031,9.732035,14.661780,78.849495
supporting_invoice_count,1676.0,279.196897,91.521863,198.000000,215.000000,249.000000,311.000000,825.000000
